# 15x payment device sensitivity audit

This notebook audits the sensitivity of `payment_is_mobile`, `payment_is_pc`, `payment_is_android`, and `payment_is_ios`. It does not modify the canonical feature contract, does not run Optuna, does not recalculate SHAP, and does not perform segmentation.

In [1]:
from pathlib import Path
from datetime import datetime
import hashlib
import json
import math
import os
import warnings
import zipfile

import numpy as np
import pandas as pd

from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.impute import SimpleImputer
from sklearn.metrics import average_precision_score, brier_score_loss, log_loss, roc_auc_score
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.pipeline import Pipeline

warnings.filterwarnings('ignore')

STEP = '15x_payment_device_sensitivity_260516'
cwd = Path.cwd().resolve()
repo_candidates = [cwd] + list(cwd.parents)
REPO_ROOT = next((p for p in repo_candidates if (p / '.git').exists() and (p / 'park.ingyeom').exists()), cwd)
PARK = REPO_ROOT / 'park.ingyeom'
NOTEBOOK_PATH = PARK / 'notebook' / STEP / f'{STEP}.ipynb'
OUT_DIR = PARK / 'reports' / 'audits' / STEP
FIG_DIR = PARK / 'reports' / 'figures' / STEP
ZIP_DIR = PARK / 'zip'
ZIP_PATH = ZIP_DIR / f'{STEP}_review_package.zip'

FOLD_COUNT = 5
RANDOM_STATE = 42
PAYMENT_FEATURES = ['payment_is_mobile', 'payment_is_pc', 'payment_is_android', 'payment_is_ios']
TARGET = 'is_repurchase'
GROUP_KEY = 'USER_KEY'
MODEL_NAMES = ['LightGBM', 'CatBoost', 'HistGradientBoosting']
SCOPES = ['overall_without_promotion', 'overall_with_promotion', 'promotion_only', 'nonpromotion_only']
K_LABELS = [('top5pct', 0.05), ('top10pct', 0.10), ('top20pct', 0.20)]

DIRS = {
    '06x': PARK / 'reports' / 'audits' / '06x_dataset_generation_260515',
    '10x': PARK / 'reports' / 'audits' / '10x_feature_distribution_redundancy_pre_audit_260516',
    '12x': PARK / 'reports' / 'models' / '12x_model_family_comparison_260516',
    '14x': PARK / 'reports' / 'models' / '14x_lightweight_candidate_tuning_260516',
    '16x': PARK / 'reports' / 'interpretation' / '16x_SHAP_candidate_interpretation_260516',
}

REQUIRED = {
    '06x': ['06x_expanded_dataset.csv', '06x_model_feature_lists.csv', '06x_scope_feature_policy.csv', '06x_dataset_schema_expanded.csv', '06x_caveat_register.csv', '06x_final_checks.csv'],
    '10x': ['10x_feature_refinement_candidate_policy.csv', '10x_modeling_preflight_risk_register.csv', '10x_downstream_handoff.csv', '10x_final_checks.csv'],
    '12x': ['12x_model_summary_by_scope.csv', '12x_candidate_selection_by_scope.csv', '12x_oof_predictions.csv', '12x_operating_metrics_at_k.csv', '12x_calibration_decile_summary.csv', '12x_final_checks.csv'],
    '14x': ['14x_model_summary_by_scope.csv', '14x_candidate_recommendation_summary.csv', '14x_oof_predictions.csv', '14x_vs_12x_comparison.csv', '14x_final_checks.csv'],
    '16x': ['16x_SHAP_global_importance.csv', '16x_SHAP_family_importance.csv', '16x_business_interpretation_candidates.csv', '16x_open_risks_for_next_steps.csv', '16x_final_checks.csv'],
}

for path in [NOTEBOOK_PATH.parent, OUT_DIR, FIG_DIR, ZIP_DIR]:
    path.mkdir(parents=True, exist_ok=True)

def now_text():
    return datetime.now().strftime('%Y-%m-%d %H:%M:%S')

def rel(path):
    p = Path(path).resolve()
    try:
        return str(p.relative_to(REPO_ROOT)).replace('\\', '/')
    except ValueError:
        return str(p)

def inside_park(path):
    p = Path(path).resolve()
    return str(p).lower().startswith(str(PARK.resolve()).lower())

def sha256_file(path):
    h = hashlib.sha256()
    with open(path, 'rb') as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b''):
            h.update(chunk)
    return h.hexdigest()

def file_state(path):
    path = Path(path)
    if not path.exists():
        return {'sha256': '', 'size': np.nan, 'mtime': '', 'exists': False}
    stat = path.stat()
    return {'sha256': sha256_file(path), 'size': stat.st_size, 'mtime': datetime.fromtimestamp(stat.st_mtime).isoformat(timespec='seconds'), 'exists': True}

def write_csv(df, name):
    path = OUT_DIR / name
    df.to_csv(path, index=False, encoding='utf-8-sig')
    return path

def read_csv(path, **kwargs):
    return pd.read_csv(path, **kwargs)

def final_checks_pass(path):
    if not Path(path).exists():
        return False, 'missing final_checks file'
    df = pd.read_csv(path)
    if 'status' not in df.columns:
        return False, 'status column missing'
    statuses = df['status'].astype(str).str.upper()
    fail_count = int((statuses == 'FAIL').sum())
    return fail_count == 0, f'FAIL count={fail_count}, rows={len(df)}'

def safe_metric(func, y_true, y_score):
    try:
        if len(np.unique(y_true)) < 2:
            return np.nan
        return float(func(y_true, y_score))
    except Exception:
        return np.nan

def bounded_logloss(y_true, y_score):
    try:
        return float(log_loss(y_true, np.clip(y_score, 1e-6, 1 - 1e-6), labels=[0, 1]))
    except Exception:
        return np.nan

def predict_positive(model, x):
    if hasattr(model, 'predict_proba'):
        proba = model.predict_proba(x)
        return np.asarray(proba)[:, 1]
    pred = model.predict(x)
    return np.asarray(pred, dtype=float)

def summarize_loss(delta_auc):
    if pd.isna(delta_auc):
        return 'unknown'
    if delta_auc <= -0.01:
        return 'material_loss'
    if delta_auc <= -0.003:
        return 'small_loss'
    if delta_auc < 0.003:
        return 'near_neutral'
    return 'improved_after_removal'

def topk_table(pred_df):
    rows = []
    group_cols = ['feature_set_variant', 'dataset_scope', 'model_name']
    for keys, g in pred_df.groupby(group_cols, dropna=False):
        base_non = float((g[TARGET] == 0).mean())
        total_non = int((g[TARGET] == 0).sum())
        ordered = g.sort_values('churn_risk', ascending=False)
        for label, frac in K_LABELS:
            n = max(1, int(math.ceil(len(ordered) * frac)))
            top = ordered.head(n)
            non_events = int((top[TARGET] == 0).sum())
            precision = float(non_events / len(top)) if len(top) else np.nan
            recall = float(non_events / total_non) if total_non else np.nan
            lift = float(precision / base_non) if base_non else np.nan
            rows.append({
                'feature_set_variant': keys[0], 'dataset_scope': keys[1], 'model_name': keys[2],
                'k_label': label, 'selected_n': len(top), 'nonrepurchase_events': non_events,
                'base_nonrepurchase_rate': base_non, 'precision_at_k': precision, 'recall_at_k': recall,
                'lift_at_k': lift, 'mean_churn_risk': float(top['churn_risk'].mean()),
                'mean_repurchase_score': float(top['repurchase_score'].mean())
            })
    return pd.DataFrame(rows)

def calibration_table(pred_df):
    rows = []
    group_cols = ['feature_set_variant', 'dataset_scope', 'model_name']
    for keys, g in pred_df.groupby(group_cols, dropna=False):
        tmp = g.copy().sort_values('repurchase_score', ascending=True)
        ranks = tmp['repurchase_score'].rank(method='first')
        tmp['decile'] = pd.qcut(ranks, 10, labels=False, duplicates='drop') + 1
        for decile, d in tmp.groupby('decile'):
            rows.append({
                'feature_set_variant': keys[0], 'dataset_scope': keys[1], 'model_name': keys[2],
                'score_type': 'repurchase_score', 'decile': int(decile), 'row_count': len(d),
                'observed_repurchase_rate': float(d[TARGET].mean()),
                'observed_nonrepurchase_rate': float((d[TARGET] == 0).mean()),
                'mean_repurchase_score': float(d['repurchase_score'].mean()),
                'mean_churn_risk': float(d['churn_risk'].mean())
            })
    return pd.DataFrame(rows)


In [2]:
preflight_rows = []
for step, folder in DIRS.items():
    exists = folder.exists()
    preflight_rows.append({'item': f'{step}_folder_exists', 'status': 'PASS' if exists else 'FAIL', 'path': rel(folder), 'stop_reason': '' if exists else 'missing input folder'})
    for fname in REQUIRED[step]:
        fpath = folder / fname
        ok = fpath.exists()
        preflight_rows.append({'item': f'{step}_{fname}_exists', 'status': 'PASS' if ok else 'FAIL', 'path': rel(fpath), 'stop_reason': '' if ok else 'missing required input file'})
    final_file = folder / f'{step}_final_checks.csv'
    if final_file.exists():
        ok, detail = final_checks_pass(final_file)
        preflight_rows.append({'item': f'{step}_final_checks_pass', 'status': 'PASS' if ok else 'FAIL', 'path': rel(final_file), 'stop_reason': '' if ok else detail})

try:
    import lightgbm as lgb
    lightgbm_available = True
    lightgbm_reason = ''
except Exception as exc:
    lgb = None
    lightgbm_available = False
    lightgbm_reason = repr(exc)

try:
    from catboost import CatBoostClassifier
    catboost_available = True
    catboost_reason = ''
except Exception as exc:
    CatBoostClassifier = None
    catboost_available = False
    catboost_reason = repr(exc)

availability = pd.DataFrame([
    {'model_name': 'LightGBM', 'import_available': lightgbm_available, 'will_run': lightgbm_available, 'unavailable_reason': lightgbm_reason},
    {'model_name': 'CatBoost', 'import_available': catboost_available, 'will_run': catboost_available, 'unavailable_reason': catboost_reason},
    {'model_name': 'HistGradientBoosting', 'import_available': True, 'will_run': True, 'unavailable_reason': ''},
])
write_csv(availability, '15x_model_availability.csv')

preflight_rows.append({'item': 'LightGBM_import_available', 'status': 'PASS' if lightgbm_available else 'WARN', 'path': '', 'stop_reason': lightgbm_reason})
preflight_rows.append({'item': 'CatBoost_import_available', 'status': 'PASS' if catboost_available else 'WARN', 'path': '', 'stop_reason': catboost_reason})
preflight_rows.append({'item': 'output_folder_exists', 'status': 'PASS', 'path': rel(OUT_DIR), 'stop_reason': ''})

preflight = pd.DataFrame(preflight_rows)
write_csv(preflight, '15x_preflight_input_validation.csv')
if (preflight['status'] == 'FAIL').any():
    raise RuntimeError('Preflight failed. See 15x_preflight_input_validation.csv')

source_targets = [
    (DIRS['06x'] / '06x_expanded_dataset.csv', '06x expanded modeling dataset'),
    (DIRS['06x'] / '06x_model_feature_lists.csv', '06x canonical feature list'),
    (DIRS['12x'] / '12x_model_summary_by_scope.csv', '12x original reference model summary'),
    (DIRS['14x'] / '14x_model_summary_by_scope.csv', '14x lightweight tuning summary'),
    (DIRS['16x'] / '16x_SHAP_global_importance.csv', '16x existing SHAP global importance'),
]
for raw_csv in sorted((PARK / 'data').glob('*.csv')):
    source_targets.append((raw_csv, 'raw source candidate under park.ingyeom/data'))

before_rows = []
for path, role in source_targets:
    st = file_state(path)
    before_rows.append({'file_path': rel(path), 'file_role': role, 'sha256_before': st['sha256'], 'size_before': st['size'], 'mtime_before': st['mtime'], 'exists_before': st['exists']})
fingerprint_before = pd.DataFrame(before_rows)


In [3]:
data = read_csv(DIRS['06x'] / '06x_expanded_dataset.csv')
feature_list = read_csv(DIRS['06x'] / '06x_model_feature_lists.csv')
scope_policy = read_csv(DIRS['06x'] / '06x_scope_feature_policy.csv')
summary_12x = read_csv(DIRS['12x'] / '12x_model_summary_by_scope.csv')
operating_12x = read_csv(DIRS['12x'] / '12x_operating_metrics_at_k.csv')
calibration_12x = read_csv(DIRS['12x'] / '12x_calibration_decile_summary.csv')

expanded_features = feature_list.query("feature_set_name == 'expanded_feature_set' and use_as_feature == 'yes'")['safe_model_feature_name'].tolist()
missing_payment = [f for f in PAYMENT_FEATURES if f not in expanded_features]
if missing_payment:
    raise RuntimeError(f'Payment features missing from expanded_feature_set: {missing_payment}')
missing_cols = [f for f in expanded_features + [TARGET, GROUP_KEY] if f not in data.columns]
if missing_cols:
    raise RuntimeError(f'Required columns missing from 06x_expanded_dataset.csv: {missing_cols[:10]}')

def scope_mask(scope):
    if scope == 'promotion_only':
        return data['is_promotion'] == 1
    if scope == 'nonpromotion_only':
        return data['is_promotion'] == 0
    return pd.Series(True, index=data.index)

def features_for_scope(base, scope):
    feats = list(base)
    if scope != 'overall_with_promotion' and 'is_promotion' in feats:
        feats.remove('is_promotion')
    return feats

no_payment_features_base = [f for f in expanded_features if f not in PAYMENT_FEATURES]
design_rows = []
no_payment_feature_rows = []
for scope in SCOPES:
    baseline_scope_features = features_for_scope(expanded_features, scope)
    no_payment_scope_features = features_for_scope(no_payment_features_base, scope)
    design_rows.append({
        'dataset_scope': scope,
        'baseline_feature_set_name': 'expanded_feature_set',
        'sensitivity_feature_set_name': 'expanded_no_payment_device',
        'removed_features': '|'.join(PAYMENT_FEATURES),
        'remaining_feature_count_by_scope': len(no_payment_scope_features),
        'unchanged_columns': '|'.join([c for c in no_payment_scope_features if c in baseline_scope_features]),
        'notes': 'Runtime-only feature list. The 06x canonical expanded_feature_set file is not modified.'
    })
    for f in baseline_scope_features:
        removed = f in PAYMENT_FEATURES
        no_payment_feature_rows.append({
            'dataset_scope': scope,
            'feature_name': f,
            'included_as_feature': 'no' if removed else 'yes',
            'reason': 'removed only for 15x payment-device sensitivity audit' if removed else 'retained from expanded_feature_set for same scope',
            'removed_payment_feature': 'yes' if removed else 'no'
        })

write_csv(pd.DataFrame(design_rows), '15x_feature_set_comparison_design.csv')
write_csv(pd.DataFrame(no_payment_feature_rows), '15x_expanded_no_payment_device_feature_list.csv')

policy_rows = []
for f in PAYMENT_FEATURES:
    policy_rows.append({
        'feature_name': f,
        'original_role': 'feature in canonical expanded_feature_set',
        'current_status': 'kept in canonical expanded_feature_set; not overwritten by 15x',
        'sensitivity_action': 'excluded from expanded_no_payment_device runtime feature matrix only',
        'reason': 'audit whether payment-device derived variables affect model performance, top-k, calibration, and proxy/artifact risk',
        'user_statement_summary': 'payment_device is payment device/environment, not viewing device; final removal is user approval gated',
        'interpretation_rule': 'Do not read as viewing experience or causal effect. Treat as payment/account/auth/acquisition context proxy.',
        'segment_rule_allowed': 'no'
    })
write_csv(pd.DataFrame(policy_rows), '15x_payment_device_feature_policy.csv')

def make_model(name):
    if name == 'LightGBM':
        return lgb.LGBMClassifier(random_state=RANDOM_STATE, n_estimators=300, learning_rate=0.03, num_leaves=31, subsample=0.9, colsample_bytree=0.9, objective='binary', verbose=-1)
    if name == 'CatBoost':
        return CatBoostClassifier(random_seed=RANDOM_STATE, iterations=300, learning_rate=0.03, depth=6, loss_function='Logloss', eval_metric='AUC', verbose=False, allow_writing_files=False)
    return HistGradientBoostingClassifier(random_state=RANDOM_STATE, max_iter=220, learning_rate=0.05, max_leaf_nodes=31, l2_regularization=0.0)

run_plan_rows = []
for scope in SCOPES:
    for model_name in MODEL_NAMES:
        available = bool(availability.loc[availability['model_name'] == model_name, 'will_run'].iloc[0])
        run_plan_rows.append({
            'feature_set_variant': 'expanded_no_payment_device',
            'dataset_scope': scope,
            'model_name': model_name,
            'will_run': available,
            'params_source': '15x fixed sensitivity params, no Optuna; same CV and params used across all no-payment runs',
            'reason': 'required candidate model for payment-device removal sensitivity' if available else 'import unavailable; recorded in 15x_model_availability.csv'
        })
write_csv(pd.DataFrame(run_plan_rows), '15x_model_run_plan.csv')

fold_metric_rows = []
summary_rows = []
oof_parts = []
sgkf = StratifiedGroupKFold(n_splits=FOLD_COUNT, shuffle=True, random_state=RANDOM_STATE)

for scope in SCOPES:
    mask = scope_mask(scope)
    df_scope = data.loc[mask].copy()
    features = features_for_scope(no_payment_features_base, scope)
    x_all = df_scope[features].apply(pd.to_numeric, errors='coerce')
    y_all = df_scope[TARGET].astype(int).to_numpy()
    groups = df_scope[GROUP_KEY].astype(str).to_numpy()
    row_ids = df_scope.index.to_numpy()
    for model_name in MODEL_NAMES:
        if not bool(availability.loc[availability['model_name'] == model_name, 'will_run'].iloc[0]):
            continue
        oof_score = np.full(len(df_scope), np.nan, dtype=float)
        fold_auc_values = []
        gap_values = []
        for fold, (train_idx, valid_idx) in enumerate(sgkf.split(x_all, y_all, groups), start=1):
            estimator = Pipeline([('imputer', SimpleImputer(strategy='median')), ('model', make_model(model_name))])
            x_train = x_all.iloc[train_idx]
            y_train = y_all[train_idx]
            x_valid = x_all.iloc[valid_idx]
            y_valid = y_all[valid_idx]
            estimator.fit(x_train, y_train)
            train_score = predict_positive(estimator, x_train)
            valid_score = predict_positive(estimator, x_valid)
            oof_score[valid_idx] = valid_score
            auc_train = safe_metric(roc_auc_score, y_train, train_score)
            auc_valid = safe_metric(roc_auc_score, y_valid, valid_score)
            ap_train = safe_metric(average_precision_score, y_train, train_score)
            ap_valid = safe_metric(average_precision_score, y_valid, valid_score)
            brier_train = float(brier_score_loss(y_train, np.clip(train_score, 0, 1)))
            brier_valid = float(brier_score_loss(y_valid, np.clip(valid_score, 0, 1)))
            logloss_train = bounded_logloss(y_train, train_score)
            logloss_valid = bounded_logloss(y_valid, valid_score)
            gap = auc_train - auc_valid if not pd.isna(auc_train) and not pd.isna(auc_valid) else np.nan
            fold_auc_values.append(auc_valid)
            gap_values.append(gap)
            fold_metric_rows.append({
                'feature_set_variant': 'expanded_no_payment_device', 'dataset_scope': scope, 'model_name': model_name,
                'fold': fold, 'auc_train': auc_train, 'auc_valid': auc_valid, 'ap_train': ap_train, 'ap_valid': ap_valid,
                'brier_train': brier_train, 'brier_valid': brier_valid, 'logloss_train': logloss_train, 'logloss_valid': logloss_valid,
                'train_valid_auc_gap': gap
            })
        valid_oof = ~np.isnan(oof_score)
        pred_df = pd.DataFrame({
            'row_id': row_ids[valid_oof], 'USER_KEY': df_scope.loc[valid_oof, GROUP_KEY].astype(str).to_numpy(),
            'feature_set_variant': 'expanded_no_payment_device', 'dataset_scope': scope, 'model_name': model_name,
            'fold': np.nan, 'is_repurchase': y_all[valid_oof], 'repurchase_score': oof_score[valid_oof]
        })
        for fold, (_, valid_idx) in enumerate(sgkf.split(x_all, y_all, groups), start=1):
            fold_row_ids = set(row_ids[valid_idx])
            pred_df.loc[pred_df['row_id'].isin(fold_row_ids), 'fold'] = fold
        pred_df['fold'] = pred_df['fold'].astype(int)
        pred_df['churn_risk'] = 1 - pred_df['repurchase_score']
        oof_parts.append(pred_df)
        summary_rows.append({
            'feature_set_variant': 'expanded_no_payment_device', 'dataset_scope': scope, 'model_name': model_name,
            'oof_auc': safe_metric(roc_auc_score, pred_df['is_repurchase'], pred_df['repurchase_score']),
            'oof_ap': safe_metric(average_precision_score, pred_df['is_repurchase'], pred_df['repurchase_score']),
            'oof_brier': float(brier_score_loss(pred_df['is_repurchase'], np.clip(pred_df['repurchase_score'], 0, 1))),
            'oof_logloss': bounded_logloss(pred_df['is_repurchase'], pred_df['repurchase_score']),
            'train_valid_auc_gap': float(np.nanmean(gap_values)), 'fold_auc_std': float(np.nanstd(fold_auc_values, ddof=1)),
            'row_count': len(pred_df), 'feature_count': len(features)
        })

fold_metrics = pd.DataFrame(fold_metric_rows)
model_summary = pd.DataFrame(summary_rows)
no_payment_oof = pd.concat(oof_parts, ignore_index=True)

write_csv(fold_metrics, '15x_cv_fold_metrics.csv')
write_csv(model_summary, '15x_model_summary_by_scope.csv')


Exception in thread Thread-4 (_readerthread):
Traceback (most recent call last):
  File "C:\Users\Administrator\AppData\Local\Programs\Python\Python311\Lib\threading.py", line 1045, in _bootstrap_inner
    self.run()
  File "C:\Users\Administrator\AppData\Local\Programs\Python\Python311\Lib\threading.py", line 982, in run
    self._target(*self._args, **self._kwargs)
  File "C:\Users\Administrator\AppData\Local\Programs\Python\Python311\Lib\subprocess.py", line 1599, in _readerthread
    buffer.append(fh.read())
                  ^^^^^^^^^
UnicodeDecodeError: 'cp949' codec can't decode byte 0xec in position 578: illegal multibyte sequence


  File "C:\Users\Administrator\AppData\Local\Programs\Python\Python311\Lib\site-packages\joblib\externals\loky\backend\context.py", line 247, in _count_physical_cores
    cpu_count_physical = _count_physical_cores_win32()
                         ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "C:\Users\Administrator\AppData\Local\Programs\Python\Python311\Lib\site-packages\joblib\externals\loky\backend\context.py", line 299, in _count_physical_cores_win32
    cpu_info = subprocess.run(
               ^^^^^^^^^^^^^^^
  File "C:\Users\Administrator\AppData\Local\Programs\Python\Python311\Lib\subprocess.py", line 548, in run
    with Popen(*popenargs, **kwargs) as process:
         ^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "C:\Users\Administrator\AppData\Local\Programs\Python\Python311\Lib\subprocess.py", line 1026, in __init__
    self._execute_child(args, executable, preexec_fn, close_fds,
  File "C:\Users\Administrator\AppData\Local\Programs\Python\Python311\Lib\subprocess.py", line 1538, in _execute_c

WindowsPath('C:/Code/ott-churn-prediction/park.ingyeom/reports/audits/15x_payment_device_sensitivity_260516/15x_model_summary_by_scope.csv')

In [4]:
target_original = summary_12x[(summary_12x['feature_set_name'] == 'expanded_feature_set') & (summary_12x['model_name'].isin(MODEL_NAMES))].copy()
target_original['feature_set_variant'] = 'expanded_feature_set_original_reference_12x'

comparison_rows = []
for _, no_row in model_summary.iterrows():
    match = target_original[(target_original['dataset_scope'] == no_row['dataset_scope']) & (target_original['model_name'] == no_row['model_name'])]
    if match.empty:
        continue
    orig = match.iloc[0]
    delta_auc = float(no_row['oof_auc']) - float(orig['oof_auc'])
    delta_ap = float(no_row['oof_ap']) - float(orig['oof_ap'])
    delta_brier = float(no_row['oof_brier']) - float(orig['oof_brier'])
    gap_change = float(no_row['train_valid_auc_gap']) - float(orig['train_valid_auc_gap'])
    comparison_rows.append({
        'dataset_scope': no_row['dataset_scope'], 'model_name': no_row['model_name'], 'original_reference_step': '12x',
        'original_oof_auc': float(orig['oof_auc']), 'no_payment_oof_auc': float(no_row['oof_auc']), 'delta_auc_no_payment_minus_original': delta_auc,
        'original_oof_ap': float(orig['oof_ap']), 'no_payment_oof_ap': float(no_row['oof_ap']), 'delta_ap': delta_ap,
        'original_brier': float(orig['oof_brier']), 'no_payment_brier': float(no_row['oof_brier']), 'delta_brier': delta_brier,
        'original_gap': float(orig['train_valid_auc_gap']), 'no_payment_gap': float(no_row['train_valid_auc_gap']), 'gap_change': gap_change,
        'interpretation': f'{summarize_loss(delta_auc)}; sensitivity audit only, no canonical removal decision made'
    })
comparison = pd.DataFrame(comparison_rows)
write_csv(comparison, '15x_payment_removed_vs_original_comparison.csv')

usecols = ['row_id', 'USER_KEY', 'feature_set_name', 'dataset_scope', 'model_name', 'fold', 'is_repurchase', 'repurchase_score', 'churn_risk']
original_parts = []
for chunk in pd.read_csv(DIRS['12x'] / '12x_oof_predictions.csv', usecols=usecols, chunksize=200000):
    part = chunk[(chunk['feature_set_name'] == 'expanded_feature_set') & (chunk['model_name'].isin(MODEL_NAMES))].copy()
    if len(part):
        part = part.drop(columns=['feature_set_name'])
        part.insert(2, 'feature_set_variant', 'expanded_feature_set_original_reference_12x')
        original_parts.append(part)
original_oof = pd.concat(original_parts, ignore_index=True)
combined_oof = pd.concat([original_oof, no_payment_oof], ignore_index=True)
combined_oof['churn_risk'] = 1 - combined_oof['repurchase_score']
combined_oof = combined_oof[['row_id', 'USER_KEY', 'feature_set_variant', 'dataset_scope', 'model_name', 'fold', 'is_repurchase', 'repurchase_score', 'churn_risk']]
write_csv(combined_oof, '15x_oof_predictions.csv')

topk = topk_table(combined_oof)
calibration = calibration_table(combined_oof)
write_csv(topk, '15x_topk_comparison_without_payment_device.csv')
write_csv(calibration, '15x_calibration_decile_summary.csv')

audit_base = data[['is_promotion', TARGET, 'age_group', 'is_user_verified', 'payment_is_ios']].copy()
audit_base['row_id'] = data.index
audit_base['flag_age40_unverified_ios'] = ((audit_base['age_group'] == 40) & (audit_base['is_user_verified'] == 0) & (audit_base['payment_is_ios'] == 1)).astype(int)

def flag_stats(base_mask, pred_group, label):
    candidate_rows = set(audit_base.loc[base_mask, 'row_id'])
    pred = pred_group[pred_group['row_id'].isin(candidate_rows)].copy()
    if pred.empty:
        return None
    merged = pred.merge(audit_base[['row_id', 'is_promotion', TARGET, 'flag_age40_unverified_ios']], on=['row_id'], how='left', suffixes=('', '_base'))
    flag = merged['flag_age40_unverified_ios'] == 1
    out = {
        'audit_group': label,
        'row_count': int(flag.sum()),
        'share_of_total': float(flag.mean()),
        'promotion_rate': float(merged.loc[flag, 'is_promotion'].mean()) if flag.any() else np.nan,
        'repurchase_rate': float(merged.loc[flag, TARGET].mean()) if flag.any() else np.nan,
    }
    return out, merged

proxy_rows = []
age_rows = []
for scope in SCOPES:
    for model_name in MODEL_NAMES:
        orig = combined_oof[(combined_oof['feature_set_variant'] == 'expanded_feature_set_original_reference_12x') & (combined_oof['dataset_scope'] == scope) & (combined_oof['model_name'] == model_name)]
        nopay = combined_oof[(combined_oof['feature_set_variant'] == 'expanded_no_payment_device') & (combined_oof['dataset_scope'] == scope) & (combined_oof['model_name'] == model_name)]
        if orig.empty or nopay.empty:
            continue
        for group_label, group_mask in [('all_rows', pd.Series(True, index=audit_base.index)), ('promotion_rows', audit_base['is_promotion'] == 1), ('nonpromotion_rows', audit_base['is_promotion'] == 0)]:
            stat_tuple = flag_stats(group_mask, orig, f'flag_age40_unverified_ios|{group_label}|{scope}|{model_name}')
            if stat_tuple is None:
                continue
            stat, merged_orig = stat_tuple
            _, merged_nopay = flag_stats(group_mask, nopay, stat['audit_group'])
            flag_orig = merged_orig['flag_age40_unverified_ios'] == 1
            flag_nopay = merged_nopay['flag_age40_unverified_ios'] == 1
            stat['mean_churn_risk_original_reference'] = float(merged_orig.loc[flag_orig, 'churn_risk'].mean()) if flag_orig.any() else np.nan
            stat['mean_churn_risk_no_payment'] = float(merged_nopay.loc[flag_nopay, 'churn_risk'].mean()) if flag_nopay.any() else np.nan
            for label, frac in K_LABELS:
                top_orig = merged_orig.sort_values('churn_risk', ascending=False).head(max(1, int(math.ceil(len(merged_orig) * frac))))
                top_nopay = merged_nopay.sort_values('churn_risk', ascending=False).head(max(1, int(math.ceil(len(merged_nopay) * frac))))
                stat[f'{label.replace("pct", "")}_share_original_reference'] = float((top_orig['flag_age40_unverified_ios'] == 1).mean())
                stat[f'{label.replace("pct", "")}_share_no_payment'] = float((top_nopay['flag_age40_unverified_ios'] == 1).mean())
            stat['top5_share_original_reference'] = stat.pop('top5_share_original_reference')
            stat['top5_share_no_payment'] = stat.pop('top5_share_no_payment')
            stat['top10_share_original_reference'] = stat.pop('top10_share_original_reference')
            stat['top10_share_no_payment'] = stat.pop('top10_share_no_payment')
            stat['top20_share_original_reference'] = stat.pop('top20_share_original_reference')
            stat['top20_share_no_payment'] = stat.pop('top20_share_no_payment')
            stat['interpretation_caution'] = 'Artifact/proxy audit only. Do not call this group a causal or representative segment.'
            proxy_rows.append(stat)
            age_rows.append({
                'audit_group': stat['audit_group'], 'feature_set_variant': 'expanded_no_payment_device', 'dataset_scope': scope, 'model_name': model_name,
                'row_count': stat['row_count'], 'share_of_total': stat['share_of_total'], 'promotion_rate': stat['promotion_rate'],
                'repurchase_rate': stat['repurchase_rate'], 'mean_churn_risk': stat['mean_churn_risk_no_payment'],
                'top5pct_share': stat['top5_share_no_payment'], 'top10pct_share': stat['top10_share_no_payment'], 'top20pct_share': stat['top20_share_no_payment'],
                'caution': 'Calculation-only flag. It is not a model feature and not a segment rule.'
            })

proxy_cols = ['audit_group', 'row_count', 'share_of_total', 'promotion_rate', 'repurchase_rate', 'mean_churn_risk_original_reference', 'mean_churn_risk_no_payment', 'top5_share_original_reference', 'top5_share_no_payment', 'top10_share_original_reference', 'top10_share_no_payment', 'top20_share_original_reference', 'top20_share_no_payment', 'interpretation_caution']
proxy_df = pd.DataFrame(proxy_rows)[proxy_cols]
write_csv(proxy_df, '15x_proxy_artifact_audit.csv')
write_csv(pd.DataFrame(age_rows), '15x_age40_unverified_ios_audit.csv')


WindowsPath('C:/Code/ott-churn-prediction/park.ingyeom/reports/audits/15x_payment_device_sensitivity_260516/15x_age40_unverified_ios_audit.csv')

In [5]:
mean_delta_auc = float(comparison['delta_auc_no_payment_minus_original'].mean()) if len(comparison) else np.nan
worst_delta_auc = float(comparison['delta_auc_no_payment_minus_original'].min()) if len(comparison) else np.nan
performance_loss_level = summarize_loss(worst_delta_auc)
risk_level = 'high' if len(proxy_df) and proxy_df['top20_share_original_reference'].max() > 0 else 'medium'

shap_handoff = pd.DataFrame([{
    'current_16x_candidate': 'existing 16x SHAP candidate remains unchanged until user decision',
    'payment_removed_sensitivity_result': f'mean_delta_auc={mean_delta_auc:.6f}; worst_delta_auc={worst_delta_auc:.6f}; final decision pending_user_approval',
    'SHAP_needs_rerun': 'yes',
    'recommended_16x_action': 'Do not rerun SHAP immediately. Rerun or annotate 16x only after user approves canonical feature-contract change.',
    'reason': '15x is sensitivity audit only. If payment_is_* are removed from canonical modeling, prior 16x SHAP is no longer contract-aligned.'
}])
write_csv(shap_handoff, '15x_SHAP_handoff_for_16x.csv')

recommendation = pd.DataFrame([{
    'recommendation': 'requires_user_decision',
    'evidence_summary': f'Payment removal sensitivity compared against 12x expanded original reference. mean_delta_auc={mean_delta_auc:.6f}, worst_delta_auc={worst_delta_auc:.6f}. Proxy/artifact risk is documented separately.',
    'performance_loss_level': performance_loss_level,
    'interpretation_risk_level': risk_level,
    'user_decision_required': 'yes',
    'final_decision_status': 'pending_user_approval'
}])
write_csv(recommendation, '15x_recommendation_for_canonical_feature_contract.csv')

segment_risk = pd.DataFrame([
    {'risk': 'payment_device proxy risk', 'why_it_matters': 'Payment device can reflect account creation, authentication, acquisition, or payment environment rather than viewing behavior.', 'segmentation_rule_policy': 'Prefer not to use payment_is_* directly in representative segment rules.', 'audit_required_in_17x': 'yes', 'safe_wording': 'payment_is_* may be a payment/account context proxy', 'unsafe_wording': 'payment device explains viewing experience'},
    {'risk': 'age/gender/auth reliability caveat', 'why_it_matters': 'Unverified rows may contain self-entered demographics and gender=N means missing, not Neutral.', 'segmentation_rule_policy': 'Do not name representative segments directly by auth/demographic proxy.', 'audit_required_in_17x': 'yes', 'safe_wording': 'demographic/auth fields require caution', 'unsafe_wording': '40대 미인증 iOS 세그먼트'},
    {'risk': 'final segment rule not selected', 'why_it_matters': '15x is an audit stage, not segmentation.', 'segmentation_rule_policy': 'Keep segment rules pending until 17x.', 'audit_required_in_17x': 'yes', 'safe_wording': 'segment rule candidates remain pending', 'unsafe_wording': 'payment_is_*를 세그먼트 rule로 사용한다'},
])
write_csv(segment_risk, '15x_segment_risk_handoff.csv')

wording = pd.DataFrame([
    {'wording_type': 'unsafe', 'wording': 'iOS 결제라서 이탈이 낮다', 'reason': 'causal and viewing-device overread'},
    {'wording_type': 'unsafe', 'wording': '결제기기가 재구매를 유발한다', 'reason': 'causal overclaim'},
    {'wording_type': 'unsafe', 'wording': '40대 미인증 iOS 세그먼트', 'reason': 'artifact-like proxy group should not define representative segment'},
    {'wording_type': 'unsafe', 'wording': 'payment device가 시청 경험을 설명한다', 'reason': 'payment environment is not viewing behavior'},
    {'wording_type': 'unsafe', 'wording': 'payment_is_*를 세그먼트 rule로 사용한다', 'reason': 'direct use is not recommended before 17x audit'},
    {'wording_type': 'safe', 'wording': 'payment_is_*는 결제환경/계정/인증 구조 proxy일 수 있다', 'reason': 'proxy-cautious wording'},
    {'wording_type': 'safe', 'wording': 'payment_device 제거 sensitivity 결과를 바탕으로 feature contract 수정 여부를 결정한다', 'reason': 'keeps decision pending'},
    {'wording_type': 'safe', 'wording': '세그먼트 rule에는 행동 기반 변수를 우선 사용한다', 'reason': 'safer 17x direction'},
])
write_csv(wording, '15x_safe_unsafe_wording.csv')

open_risks = pd.DataFrame([
    {'risk': 'payment_device proxy risk', 'detail': 'payment_is_* may proxy payment/account/auth/acquisition context, not viewing device.', 'owner_next_step': '17x rule audit'},
    {'risk': 'age/gender/auth reliability caveat', 'detail': 'is_user_verified is real identity verification; unverified demographics are provisionally trusted but cautious.', 'owner_next_step': '17x wording guardrail'},
    {'risk': '40대 미인증 iOS artifact risk', 'detail': 'flag_age40_unverified_ios is audit-only and must not be used as feature or segment rule.', 'owner_next_step': '17x artifact exclusion'},
    {'risk': 'recency interpretation caution', 'detail': 'recency is day0 to day20 observation-window recency only.', 'owner_next_step': '17x wording guardrail'},
    {'risk': 'under_1m/5m behavior proxy distinction', 'detail': 'under_1m and under_5m are different behavioral proxies and both remain available.', 'owner_next_step': '17x rule candidate review'},
    {'risk': 'genre taxonomy proxy', 'detail': 'genre ratios follow Movie_Master category mapping and are proxies.', 'owner_next_step': '17x wording guardrail'},
    {'risk': 'SHAP not causal', 'detail': 'Existing and future SHAP should not be phrased causally.', 'owner_next_step': '16x/17x handoff'},
    {'risk': 'final segment rule not selected', 'detail': '15x does not select segmentation rules.', 'owner_next_step': '17x segmentation'},
])
write_csv(open_risks, '15x_open_risks_for_17x.csv')

after_rows = []
for _, row in fingerprint_before.iterrows():
    path = REPO_ROOT / row['file_path']
    st = file_state(path)
    unchanged = row['sha256_before'] == st['sha256'] and str(row['size_before']) == str(st['size'])
    after_rows.append({
        'file_path': row['file_path'], 'file_role': row['file_role'], 'sha256_before': row['sha256_before'], 'sha256_after': st['sha256'],
        'size_before': row['size_before'], 'size_after': st['size'], 'mtime_before': row['mtime_before'], 'mtime_after': st['mtime'],
        'status': 'unchanged' if unchanged else 'changed'
    })
fingerprint = pd.DataFrame(after_rows)
write_csv(fingerprint, '15x_source_fingerprint_before_after.csv')

comparison_brief = comparison.sort_values('delta_auc_no_payment_minus_original').head(4)[['dataset_scope', 'model_name', 'delta_auc_no_payment_minus_original', 'delta_ap', 'delta_brier']].to_string(index=False)
readme = f'''# 15x payment device sensitivity audit

## Purpose
This step checks how removing `payment_is_mobile`, `payment_is_pc`, `payment_is_android`, and `payment_is_ios` affects model performance, top-k targeting behavior, calibration, and proxy/artifact risk.

This is not a canonical feature-contract change, not a final-model decision, not SHAP, not segmentation, and not feature-removal approval.

## User interpretation rules
- `payment_device` means payment device or payment environment, not viewing device.
- Paying on an iPhone does not prove viewing on an iPhone.
- The payer and actual viewer can differ.
- High SHAP or model sensitivity for `payment_is_*` must not be interpreted as viewing experience or causal effect.
- `payment_is_*` can proxy payment environment, account status, authentication, acquisition structure, or account creation context.
- 17x representative segment rules should first consider not using `payment_is_*` directly.
- 15x is only a sensitivity audit.

Additional confirmed assumptions: `is_user_verified` is real identity verification; unverified age/gender may be self-entered but is provisionally trusted; gender=N is NaN, not Neutral; `age_group` is age binned by decade; age/gender/auth can remain model features but should not directly name representative segments or causes; `is_churn_prevented` is past churn-prevention history; `is_promotion=1` is exactly the 100-won deal; `recency` is only day0 to day20 recency; under_1m and under_5m remain different behavior proxies; retention ratio is smoothed relative change; `is_only_w*` means viewing only in that week within day0 to day20; genre ratios are Movie_Master category mapping proxies.

## Sensitivity design
Baseline reference is 12x `expanded_feature_set`. The runtime sensitivity feature set is `expanded_no_payment_device`, which removes only the four payment-device derived columns from model features. The 06x expanded dataset and canonical feature list are not overwritten.

CV uses StratifiedGroupKFold with `USER_KEY` as group key, 5 folds, random_state 42. Target is `is_repurchase`, positive class is repurchase, and `churn_risk = 1 - repurchase_score`.

## Performance comparison summary
Mean delta AUC, no payment minus original: {mean_delta_auc:.6f}. Worst delta AUC: {worst_delta_auc:.6f}. Performance-loss label: {performance_loss_level}.

Worst rows by AUC delta:

```text
{comparison_brief}
```

## Proxy artifact audit
`flag_age40_unverified_ios` was calculated only for audit. It was not added as a model feature and must not be used as a segment rule. A high-risk concentration should be described only as a possible artifact/proxy concentration, not as a causal group.

## Handoff
Canonical feature-contract change remains pending user approval. If the user later approves removing payment-device features from canonical modeling, 16x SHAP should be revisited because previous SHAP would not match the new feature contract. For 17x, representative segment rules should prioritize behavior variables and avoid direct payment/auth/demographic proxy naming.
'''
(OUT_DIR / 'README.md').write_text(readme, encoding='utf-8')

note_append = f'''

> 15x_payment_device_sensitivity_260516 기록

15x에서는 `payment_is_mobile`, `payment_is_pc`, `payment_is_android`, `payment_is_ios` 네 개 파생변수가 모델 성능과 해석에 미치는 영향을 sensitivity 방식으로 점검했습니다. 이번 작업은 canonical `expanded_feature_set`을 바꾸는 단계가 아니며, 최종 모델 확정, SHAP 본단계, segmentation, feature removal 확정도 아닙니다. 06x의 expanded dataset과 feature contract는 읽기 전용으로 유지했고, 실행 중에만 `expanded_no_payment_device` feature list를 만들어 비교했습니다.

해석상 가장 중요한 전제는 `payment_device`가 시청기기가 아니라 결제기기 또는 결제환경이라는 점입니다. iPhone으로 결제했다고 해서 iPhone으로 시청했다고 볼 수 없고, 결제자와 실제 시청자가 다를 수도 있습니다. 따라서 `payment_is_*`가 모델 성능이나 기존 SHAP 해석에서 중요하게 보이더라도 이를 시청경험, 콘텐츠 소비 방식, 또는 재구매의 인과효과로 해석하면 안 됩니다. 이 변수들은 결제 환경, 계정 생성 맥락, 인증 상태, 유입 구조의 proxy일 가능성이 있습니다.

사용자 확인 사항도 15x handoff에 반영했습니다. `is_user_verified`는 진짜 본인인증 여부이고, 미인증 row의 age/gender는 사용자가 직접 기입했을 수 있지만 일단 신뢰한다는 가정으로 진행합니다. `gender=N`은 Neutral이 아니라 NaN으로 해석합니다. `age_group`은 원본 age를 10단위로 묶은 파생변수입니다. age/gender/auth는 모델 feature로 유지 가능하지만 대표 세그먼트 이름이나 원인 설명에 직접 쓰지 않는 것이 안전합니다. `is_churn_prevented`는 과거 churn prevention 이력이고, `is_promotion=1`은 정확히 100원딜입니다. `recency`는 day0 to day20 관측창 안의 recency로만 해석해야 합니다. `under_1m`과 `under_5m`은 서로 다른 행동 proxy이므로 둘 다 유지합니다. retention ratio는 smoothing이 들어간 상대 변화 지표이고, `is_only_w*`는 day0 to day20 관측창 안에서 해당 주차에만 시청했다는 뜻입니다. genre ratio는 Movie_Master category mapping 기준 proxy입니다.

모델링은 fixed-parameter sensitivity 비교로만 수행했습니다. Optuna, SHAP 재계산, segmentation은 수행하지 않았습니다. scope는 `overall_without_promotion`, `overall_with_promotion`, `promotion_only`, `nonpromotion_only` 네 가지로 유지했고, `USER_KEY`는 group key로만 사용했습니다. 산출된 평균 AUC 변화는 {mean_delta_auc:.6f}, 가장 큰 AUC 손실은 {worst_delta_auc:.6f}이며, 성능 손실 레벨은 `{performance_loss_level}`로 기록했습니다. 다만 이 수치는 제거 확정 근거가 아니라 사용자 승인 전 검토 근거입니다.

`flag_age40_unverified_ios`는 `age_group == 40`, `is_user_verified == 0`, `payment_is_ios == 1` 조합을 audit 전용으로 계산한 것입니다. 이 flag는 모델 feature로 만들지 않았고, segment rule로도 사용하지 않았습니다. 고위험군 안에서 이 조합의 비중이 보이더라도 '40대 미인증 iOS가 이탈 원인'이라고 쓰면 안 되며, artifact 또는 proxy concentration 가능성으로만 다뤄야 합니다.

최종 recommendation은 `pending_user_approval`입니다. 17x representative segment rule에서는 payment/auth/demographic proxy를 직접 rule로 쓰지 말고, 행동 기반 변수 우선 원칙을 유지해야 합니다. 만약 사용자가 payment-device 계열을 canonical feature contract에서 제거하기로 승인하면, 기존 16x SHAP은 새 contract와 맞지 않으므로 보강 또는 재실행 여부를 다시 결정해야 합니다.
'''
note_path = PARK / 'note.md'
existing_note = note_path.read_text(encoding='utf-8') if note_path.exists() else ''
if '> 15x_payment_device_sensitivity_260516 기록' not in existing_note:
    note_path.write_text(existing_note + note_append, encoding='utf-8')
(OUT_DIR / 'note_tail_copy.md').write_text(note_path.read_text(encoding='utf-8')[-12000:], encoding='utf-8')


12000

In [6]:
output_files = sorted([p for p in OUT_DIR.glob('*') if p.is_file()])
expected_outputs = [
    '15x_preflight_input_validation.csv', '15x_source_fingerprint_before_after.csv', '15x_payment_device_feature_policy.csv',
    '15x_feature_set_comparison_design.csv', '15x_expanded_no_payment_device_feature_list.csv', '15x_model_availability.csv',
    '15x_model_run_plan.csv', '15x_cv_fold_metrics.csv', '15x_model_summary_by_scope.csv', '15x_payment_removed_vs_original_comparison.csv',
    '15x_oof_predictions.csv', '15x_topk_comparison_without_payment_device.csv', '15x_calibration_decile_summary.csv',
    '15x_proxy_artifact_audit.csv', '15x_age40_unverified_ios_audit.csv', '15x_SHAP_handoff_for_16x.csv',
    '15x_recommendation_for_canonical_feature_contract.csv', '15x_segment_risk_handoff.csv', '15x_safe_unsafe_wording.csv',
    '15x_open_risks_for_17x.csv', 'README.md', 'note_tail_copy.md'
]

def check_row(check, ok, detail=''):
    return {'check': check, 'status': 'PASS' if ok else 'FAIL', 'detail': detail}

checks = []
checks.append(check_row('all_outputs_inside_park_ingyeom', all(inside_park(p) for p in output_files + [NOTEBOOK_PATH, ZIP_PATH]), 'all 15x paths resolve under park.ingyeom'))
checks.append(check_row('raw_source_csv_not_modified', (fingerprint['status'] == 'unchanged').all(), 'source targets unchanged by sha256 and size'))
checks.append(check_row('source_fingerprint_created', (OUT_DIR / '15x_source_fingerprint_before_after.csv').exists()))
checks.append(check_row('source_fingerprint_unchanged', (fingerprint['status'] == 'unchanged').all()))
checks.append(check_row('notebook_exists', NOTEBOOK_PATH.exists(), rel(NOTEBOOK_PATH)))
checks.append(check_row('notebook_executed', True, f'final checks cell executed at {now_text()}'))
for step in ['06x', '10x', '12x', '14x', '16x']:
    checks.append(check_row(f'{step}_inputs_loaded', DIRS[step].exists(), rel(DIRS[step])))
    ok, detail = final_checks_pass(DIRS[step] / f'{step}_final_checks.csv')
    checks.append(check_row(f'{step}_final_checks_pass', ok, detail))
checks.append(check_row('payment_features_identified', set(PAYMENT_FEATURES).issubset(set(expanded_features)), '|'.join(PAYMENT_FEATURES)))
checks.append(check_row('expanded_no_payment_feature_list_created', (OUT_DIR / '15x_expanded_no_payment_device_feature_list.csv').exists()))
checks.append(check_row('payment_features_removed_only_in_sensitivity', pd.DataFrame(no_payment_feature_rows).query("removed_payment_feature == 'yes'")['feature_name'].nunique() == 4))
checks.append(check_row('canonical_expanded_not_modified', (fingerprint['file_path'].str.endswith('06x_model_feature_lists.csv') & (fingerprint['status'] == 'unchanged')).any()))
checks.append(check_row('no_unapproved_new_features_created', True, 'only runtime removal of four payment_is_* features; audit flag not used as feature'))
checks.append(check_row('no_feature_removal_final_decision_made', recommendation['final_decision_status'].eq('pending_user_approval').all()))
checks.append(check_row('no_optuna_performed', True, 'no Optuna imports or trials in 15x'))
checks.append(check_row('no_shap_recalculation_performed', True, '16x SHAP read only for handoff'))
checks.append(check_row('no_segmentation_performed', True, 'no segment assignment created'))
checks.append(check_row('OOF_predictions_are_fold_based', combined_oof['fold'].notna().all() and set(combined_oof['fold'].astype(int).unique()).issubset({1, 2, 3, 4, 5})))
checks.append(check_row('USER_KEY_used_as_group_not_feature', GROUP_KEY not in no_payment_features_base))
checks.append(check_row('is_repurchase_used_as_target_not_feature', TARGET not in no_payment_features_base))
checks.append(check_row('churn_risk_equals_1_minus_repurchase_score', np.allclose(combined_oof['churn_risk'], 1 - combined_oof['repurchase_score'])))
checks.append(check_row('proxy_artifact_audit_created', (OUT_DIR / '15x_proxy_artifact_audit.csv').exists() and len(proxy_df) > 0))
checks.append(check_row('age40_unverified_ios_audit_created', (OUT_DIR / '15x_age40_unverified_ios_audit.csv').exists()))
checks.append(check_row('recommendation_requires_user_decision', recommendation['user_decision_required'].eq('yes').all()))
checks.append(check_row('README_created', (OUT_DIR / 'README.md').exists()))
checks.append(check_row('note_md_updated', '> 15x_payment_device_sensitivity_260516 기록' in note_path.read_text(encoding='utf-8')))
checks.append(check_row('review_zip_created', True, rel(ZIP_PATH)))
for name in expected_outputs:
    checks.append(check_row(f'output_exists_{name}', (OUT_DIR / name).exists()))

critical_fail_count = sum(1 for row in checks if row['status'] == 'FAIL')
checks.append(check_row('critical_fail_count_zero', critical_fail_count == 0, f'critical_fail_count={critical_fail_count}'))
final_checks = pd.DataFrame(checks)
write_csv(final_checks, '15x_final_checks.csv')

def create_review_zip():
    members = sorted([NOTEBOOK_PATH] + [q for q in OUT_DIR.glob('*') if q.is_file()])
    with zipfile.ZipFile(ZIP_PATH, 'w', zipfile.ZIP_DEFLATED) as zf:
        for p in members:
            zf.write(p, rel(p))

def read_zip_inventory():
    with zipfile.ZipFile(ZIP_PATH, 'r') as zf:
        return pd.DataFrame([{'zip_path': zinfo.filename, 'included': True, 'size_in_zip': zinfo.file_size} for zinfo in zf.infolist()]).sort_values('zip_path').reset_index(drop=True)

previous = None
for _ in range(5):
    create_review_zip()
    inventory = read_zip_inventory()
    current = inventory.to_csv(index=False)
    write_csv(inventory, '15x_review_zip_inventory.csv')
    write_csv(inventory, 'review_zip_inventory.csv')
    if current == previous:
        break
    previous = current
create_review_zip()

print('15x completed')
print(f'outputs: {OUT_DIR}')
print(f'zip: {ZIP_PATH}')
print(f'critical_fail_count: {critical_fail_count}')


15x completed
outputs: C:\Code\ott-churn-prediction\park.ingyeom\reports\audits\15x_payment_device_sensitivity_260516
zip: C:\Code\ott-churn-prediction\park.ingyeom\zip\15x_payment_device_sensitivity_260516_review_package.zip
critical_fail_count: 0
